# 데이터 불러오기

In [2]:
# 필수 통계 라이브러리 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
from scipy import stats

In [3]:
# 한국어 폰트 설정
import platform, matplotlib
if platform.system() == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    matplotlib.rc('font', family='Malgun Gothic')
else:
    # Linux / Colab
    matplotlib.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

In [4]:
# 경고 억제 및 출력 설정
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

In [5]:
# 1. 파일 불러오기 (기본)
# portfolio.csv
df_portfolio = pd.read_csv('Portfolio_clean.csv')

# profile.csv
df_profile = pd.read_csv('profile_clean.csv')

# starbucks_menu_260112.csv
df_menus = pd.read_csv('starbucks_menu_clean.csv')

# transcript.csv (용량이 클 경우 대비)
df_transcript = pd.read_csv('transcript_clean.csv')

# 데이터 구조 확인

In [7]:
# 2. 데이터 확인 (첫 5행)
df_portfolio.head()

,reward,social,web,mobile,email,difficulty,duration,offer_type,promo_id
0,10,1,0,1,1,10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,1,1,1,1,10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,0,1,1,1,0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,0,1,1,1,5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,0,1,0,1,20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [9]:
df_transcript.head()

,customer_id,event,Record,Record Value,Reward,time
0,e078183556ad4ef3902aa1d963e61314,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558
1,e1469309339d48f299c1431d6de4eaa6,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558
2,e1c5485f813a47a984401e3f8e618d40,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558
3,e89dce9d75634dc5bf1df0a0cea0f316,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558
4,eeed22e7944b4eacb50866dc064a0f1d,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558


In [10]:
df_profile.head()

,gender,age,customer_id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,12/02/2017,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,15/07/2017,112000.0000
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,12/07/2018,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,09/05/2017,100000.0000
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,04/08/2017,NaN


In [18]:
import pandas as pd

merged_df = pd.merge(df_transcript, df_profile, on='customer_id', how='left')

final_df = pd.merge(merged_df, df_portfolio, left_on='Record Value', right_on='promo_id', how='left')

print(f"원본 데이터(transcript) 행 개수: {df_transcript.shape[0]:,}")
print(f"병합된 데이터(final_df) 행 개수: {final_df.shape[0]:,}")

# 중복되는 조인 키 promo_id 컬럼 제거
if 'promo_id' in final_df.columns:
    final_df.drop('promo_id', axis=1, inplace=True)

final_df.head()


원본 데이터(transcript) 행 개수: 306,534
병합된 데이터(final_df) 행 개수: 306,534


,customer_id,event,Record,Record Value,Reward,time,gender,age,became_member_on,income,reward,social,web,mobile,email,difficulty,duration,offer_type
0,e078183556ad4ef3902aa1d963e61314,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558,F,50,01/01/2017,98000.0000,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,3.0000,informational
1,e1469309339d48f299c1431d6de4eaa6,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558,F,55,03/01/2018,83000.0000,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,3.0000,informational
2,e1c5485f813a47a984401e3f8e618d40,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558,NaN,118,31/08/2015,NaN,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,3.0000,informational
3,e89dce9d75634dc5bf1df0a0cea0f316,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558,M,46,23/07/2018,40000.0000,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,3.0000,informational
4,eeed22e7944b4eacb50866dc064a0f1d,offer viewed,offer_id,5a8bc65990b245e5a138643cd4eb9837,NaN,558,NaN,118,20/11/2015,NaN,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,3.0000,informational


In [19]:
# 1. 오퍼(Record Value)별 수신(Received) 및 완료(Completed) 횟수 계산
# event 컬럼에서 필요한 이벤트만 필터링한 후, 이벤트 종류별로 카운팅합니다.
event_counts = final_df[final_df['event'].isin(['offer received', 'offer completed'])]
event_counts = event_counts.groupby(['Record Value', 'event']).size().unstack(fill_value=0)

# 알아보기 쉽게 컬럼명 변경
event_counts.columns.name = None  # 축 이름 제거
event_counts = event_counts[['offer received', 'offer completed']] # 순서 고정
event_counts.rename(columns={
    'offer received': 'count_received', 
    'offer completed': 'count_completed'
}, inplace=True)

# 2. 오퍼 완료 시점에 달성한 Difficulty와 지급된 Reward의 총합 계산
# 회사 관점에서 비용(Reward)이 나가고 매출(Difficulty)이 발생한 것은 '완료(Completed)'된 시점이므로 필터링합니다.
completed_df = final_df[final_df['event'] == 'offer completed']

# 포트폴리오에서 병합된 difficulty와 reward 컬럼을 sum() 처리
# 주의: 보유하신 컬럼명 대소문자에 따라 'Difficulty' 등 확인 후 수정하세요.
sums_df = completed_df.groupby('Record Value')[['difficulty', 'reward']].sum()

# 3. 데이터프레임 하나로 합치기
kpi_df = event_counts.join(sums_df).fillna(0)

# 4. KPI 수식 구현
# KPI = (sum(Difficulty) - sum(Reward)) * ( count(Completed) / count(Received) )
kpi_df['net_profit'] = kpi_df['difficulty'] - kpi_df['reward']  # (sum_Difficulty - sum_Reward)
kpi_df['conversion_rate'] = kpi_df['count_completed'] / kpi_df['count_received'] # (count_Completed / count_Received)

# 최종 KPI 도출
kpi_df['KPI_Score'] = kpi_df['net_profit'] * kpi_df['conversion_rate']

# 결과 확인 (KPI가 가장 높은 기준 내림차순 정렬)
kpi_df = kpi_df.sort_values(by='KPI_Score', ascending=False)
display(kpi_df)


,count_received,count_completed,difficulty,reward,net_profit,conversion_rate,KPI_Score
Record Value,,,,,,,
fafdcd668e3743c1bb461111dcafc2a4,7597,5317,53170.0000,10634.0000,42536.0000,0.6999,29770.1609
0b1e1539f2cc45b7b9fa7c272da2e1d7,7668,3420,68400.0000,17100.0000,51300.0000,0.4460,22880.2817
2906b810c7d4411798c6938adc9daaa5,7632,4017,40170.0000,8034.0000,32136.0000,0.5263,16914.3491
2298d6c36e964ae4a3e7e9706d1fb8c2,7646,5156,36092.0000,15468.0000,20624.0000,0.6743,13907.5783
4d5c57ea9a6940dd891ad53e9dbe8da0,7593,3331,33310.0000,33310.0000,0.0000,0.4387,0.0000
3f207df678b143eea3cee63160fa8bed,7617,0,0.0000,0.0000,0.0000,0.0000,0.0000
5a8bc65990b245e5a138643cd4eb9837,7618,0,0.0000,0.0000,0.0000,0.0000,0.0000
9b98b8c7a33c4b65b9aebfe6a799e6d9,7677,4354,21770.0000,21770.0000,0.0000,0.5671,0.0000
ae264e3637204a6fb9bb56bc8210ddfd,7658,3688,36880.0000,36880.0000,0.0000,0.4816,0.0000


In [20]:
# 1. Informational 오퍼를 제외한 새로운 DF 생성
target_df = final_df[final_df['offer_type'] != 'informational'].copy()

# 2. 오퍼 타입(offer_type)별 수신(Received) 및 완료(Completed) 횟수 계산
event_counts = target_df[target_df['event'].isin(['offer received', 'offer completed'])]

# 이번에는 기준(groupby)을 'Record Value(오퍼ID)'가 아닌 'offer_type'으로 잡습니다.
event_counts = event_counts.groupby(['offer_type', 'event']).size().unstack(fill_value=0)

# 알아보기 쉽게 컬럼명 정리
event_counts.columns.name = None
event_counts = event_counts[['offer received', 'offer completed']] 
event_counts.rename(columns={
    'offer received': 'count_received', 
    'offer completed': 'count_completed'
}, inplace=True)

# 3. 완료 시점의 Difficulty 합계와 Reward 합계를 오퍼 타입별로 계산
completed_df = target_df[target_df['event'] == 'offer completed']
sums_df = completed_df.groupby('offer_type')[['difficulty', 'reward']].sum()

# 4. 데이터 병합 및 KPI 연산
kpi_type_df = event_counts.join(sums_df).fillna(0)

# (sum(Difficulty) - sum(Reward))
kpi_type_df['net_profit'] = kpi_type_df['difficulty'] - kpi_type_df['reward']  
# (count(Completed) / count(Received))
kpi_type_df['conversion_rate'] = kpi_type_df['count_completed'] / kpi_type_df['count_received'] 

# 최종 KPI 도출
kpi_type_df['KPI_Score'] = kpi_type_df['net_profit'] * kpi_type_df['conversion_rate']

# KPI 점수가 높은 순(내림차순) 정렬 후 출력
kpi_type_df = kpi_type_df.sort_values(by='KPI_Score', ascending=False)
display(kpi_type_df)


,count_received,count_completed,difficulty,reward,net_profit,conversion_rate,KPI_Score
offer_type,,,,,,,
discount,30543,17910,197832.0000,51236.0000,146596.0000,0.5864,85961.9016
bogo,30499,15669,113440.0000,113440.0000,0.0000,0.5138,0.0000


In [21]:
# 1. 오퍼(Record Value)별 수신(Received) 및 완료(Completed) 횟수 계산
# event 컬럼에서 필요한 이벤트만 필터링한 후, 이벤트 종류별로 카운팅합니다.
event_counts = target_df[target_df['event'].isin(['offer received', 'offer completed'])]
event_counts = event_counts.groupby(['Record Value', 'event']).size().unstack(fill_value=0)

# 알아보기 쉽게 컬럼명 변경
event_counts.columns.name = None  # 축 이름 제거
event_counts = event_counts[['offer received', 'offer completed']] # 순서 고정
event_counts.rename(columns={
    'offer received': 'count_received', 
    'offer completed': 'count_completed'
}, inplace=True)

# 2. 오퍼 완료 시점에 달성한 Difficulty와 지급된 Reward의 총합 계산
# 회사 관점에서 비용(Reward)이 나가고 매출(Difficulty)이 발생한 것은 '완료(Completed)'된 시점이므로 필터링합니다.
completed_df = final_df[final_df['event'] == 'offer completed']

# 포트폴리오에서 병합된 difficulty와 reward 컬럼을 sum() 처리
# 주의: 보유하신 컬럼명 대소문자에 따라 'Difficulty' 등 확인 후 수정하세요.
sums_df = completed_df.groupby('Record Value')[['difficulty', 'reward']].sum()

# 3. 데이터프레임 하나로 합치기
kpi_df = event_counts.join(sums_df).fillna(0)

# 4. KPI 수식 구현
# KPI = (sum(Difficulty) - sum(Reward)) * ( count(Completed) / count(Received) )
kpi_df['net_profit'] = kpi_df['difficulty'] - kpi_df['reward']  # (sum_Difficulty - sum_Reward)
kpi_df['conversion_rate'] = kpi_df['count_completed'] / kpi_df['count_received'] # (count_Completed / count_Received)

# 최종 KPI 도출
kpi_df['KPI_Score'] = kpi_df['net_profit'] * kpi_df['conversion_rate']

# 결과 확인 (KPI가 가장 높은 기준 내림차순 정렬)
kpi_df = kpi_df.sort_values(by='KPI_Score', ascending=False)
display(kpi_df)


,count_received,count_completed,difficulty,reward,net_profit,conversion_rate,KPI_Score
Record Value,,,,,,,
fafdcd668e3743c1bb461111dcafc2a4,7597,5317,53170.0000,10634.0000,42536.0000,0.6999,29770.1609
0b1e1539f2cc45b7b9fa7c272da2e1d7,7668,3420,68400.0000,17100.0000,51300.0000,0.4460,22880.2817
2906b810c7d4411798c6938adc9daaa5,7632,4017,40170.0000,8034.0000,32136.0000,0.5263,16914.3491
2298d6c36e964ae4a3e7e9706d1fb8c2,7646,5156,36092.0000,15468.0000,20624.0000,0.6743,13907.5783
4d5c57ea9a6940dd891ad53e9dbe8da0,7593,3331,33310.0000,33310.0000,0.0000,0.4387,0.0000
9b98b8c7a33c4b65b9aebfe6a799e6d9,7677,4354,21770.0000,21770.0000,0.0000,0.5671,0.0000
ae264e3637204a6fb9bb56bc8210ddfd,7658,3688,36880.0000,36880.0000,0.0000,0.4816,0.0000
f19421c1d4aa40978ebb69ca19b0e20d,7571,4296,21480.0000,21480.0000,0.0000,0.5674,0.0000
